# Phase 2 -- Train the Unified PersonaPlex Emotion Network

**PersonaPlex in-house emotion network project** -- this notebook trains the single lightweight emotion head that will **replace the user-voice emotion2vec path** in the live system: it reads PersonaPlex's own Helium hidden states (the same stream the live server already captures) and publishes the emotion of the reply the model is *actually producing*.

## What this notebook does

1. Loads the Phase 1 dataset (`phase1_dataset.json`, `windows_index.csv`, packed sessions) and the Phase 0 recommendation (which representation input + head family probed best).
2. Builds **leakage-safe splits** (by `session_id`; windows from one session never cross splits) plus a **voice-held-out** generalization test.
3. Trains the unified head on **expressed-emotion labels** (human-reviewed where available), with optional soft-label distillation from the preserved teacher score vectors. Class-weighted loss, early stopping, full checkpoint/resume.
4. Evaluates: accuracy, macro/per-class F1, confusion, per-voice metrics, leave-one-voice-out generalization, and the auxiliary neutral-vs-emotional score the live path cares about.
5. **Calibrates** the head (temperature scaling fitted on validation; ECE reported before/after) -- the live system consumes confidence, not just argmax.
6. **Validates the live contract**: simulates the live deque (100-step window + 6-step lookahead at 12.5 Hz), measures GPU/CPU latency (p50/p95) against the 80 ms/step budget, measures VRAM, and exports fp16.
7. Exports deployment artifacts + the integration contract showing exactly where the live server swaps the user-voice emotion source for this head (downstream `EmotionConditioner` -> FM CFG -> renderer path untouched).

## Inputs / outputs

| | Path |
|---|---|
| Input | `/workspace/persona_emotion/phase1/` (dataset), `/workspace/persona_emotion/phase0/results/phase0_results.json` (recommendation) |
| Output | `/workspace/persona_emotion/phase2/` -- `unified_emotion_head.pt`, `unified_emotion_head_config.json`, `phase2_results.json`, `phase2_report.md`, `plots/` |

**Prerequisites:** Phase 1 complete (windows labeled; review queue ideally cleared).

In [ ]:
# ============================ Configuration =================================
import os, sys, json, time, math, random, subprocess, shutil, contextlib
from pathlib import Path

WORKSPACE = Path(os.environ.get("WORKSPACE", "/workspace"))
PHASE0_DIR = Path(os.environ.get("PERSONA_EMO_PHASE0_DIR", str(WORKSPACE / "persona_emotion" / "phase0")))
PHASE1_DIR = Path(os.environ.get("PERSONA_EMO_PHASE1_DIR", str(WORKSPACE / "persona_emotion" / "phase1")))
PHASE2_DIR = Path(os.environ.get("PERSONA_EMO_PHASE2_DIR", str(WORKSPACE / "persona_emotion" / "phase2")))

MANIFEST_JSON = PHASE1_DIR / "phase1_dataset.json"
WINDOWS_INDEX_CSV = PHASE1_DIR / "windows_index.csv"
SESSIONS_DIR = PHASE1_DIR / "sessions"
PHASE0_RESULTS = PHASE0_DIR / "results" / "phase0_results.json"

# --- Head selection (Phase-0-informed; overridable) ---
PHASE0_RECO = json.loads(PHASE0_RESULTS.read_text())["recommendation"] if PHASE0_RESULTS.exists() else {}
HEAD_TYPE = os.environ.get("EMO_HEAD_TYPE", "gru")          # "gru" | "tinyth" | "mlp"
INPUT_TYPE = os.environ.get("EMO_HEAD_INPUT", "hidden_steps")  # "hidden_steps" | "hidden_mean"
HIDDEN_DIM = 4096
HEAD_HIDDEN = 256
NUM_EMOTIONS = 6
EMOTION_NAMES = {0: "neutral", 1: "happy", 2: "sad", 3: "angry", 4: "surprise", 5: "fear"}

# --- Training ---
SEED = 7
BATCH_SIZE = 32
LR = 3e-4
EPOCHS = 60
PATIENCE = 8
LOSS_MODE = "hard"        # "hard" (expressed labels) | "soft" (teacher distributions) | "hard+soft"
SOFT_SOURCE = "vocal"     # which teacher distribution to distill from in soft modes
RUN_LOVO = True           # leave-one-voice-out generalization runs (slower)
FORCE_RETRAIN = False     # True: ignore existing checkpoints and retrain
VAL_FRACTION = 0.15
TEST_FRACTION = 0.15

# --- Live contract (validated at the end) ---
WINDOW_STEPS = 100
LOOKAHEAD_STEPS = 6
STEP_HZ = 12.5
LIVE_BUDGET_MS = 10.0     # head must stay far below the 80 ms step budget

MODEL_PATH = PHASE2_DIR / "unified_emotion_head.pt"
CONFIG_PATH = PHASE2_DIR / "unified_emotion_head_config.json"
RESULTS_JSON = PHASE2_DIR / "phase2_results.json"
REPORT_MD = PHASE2_DIR / "phase2_report.md"
PLOTS_DIR = PHASE2_DIR / "plots"
CKPT_PATH = PHASE2_DIR / "train_checkpoint.pt"
for _p in (PHASE2_DIR, PLOTS_DIR):
    _p.mkdir(parents=True, exist_ok=True)

def log(msg):
    line = "[%s] %s" % (time.strftime("%H:%M:%S"), msg)
    print(line, flush=True)
    with open(PHASE2_DIR / "phase2.log", "a", encoding="utf-8") as f:
        f.write(line + "\n")

def save_json(obj, path):
    tmp = str(path) + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, default=str)
    os.replace(tmp, path)

random.seed(SEED)
import numpy as np
np.random.seed(SEED)
import torch
log("Configuration loaded. Phase 0 recommendation: %s" % (PHASE0_RECO.get("input_and_head") or "(none found)"))
log("Head: %s on %s | loss=%s | LOVO=%s" % (HEAD_TYPE, INPUT_TYPE, LOSS_MODE, RUN_LOVO))
assert MANIFEST_JSON.exists() and WINDOWS_INDEX_CSV.exists(), "Phase 1 artifacts missing -- run Phase 1 first."
manifest = json.loads(MANIFEST_JSON.read_text())
assert manifest["contract"]["hidden_dim"] == HIDDEN_DIM

In [ ]:
# ==================== Environment + light dependencies =======================
def pip_install(pkg_spec):
    r = subprocess.run("%s -m pip install --quiet %s" % (sys.executable, pkg_spec), shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-2500:]); print(r.stderr[-2500:])
        raise RuntimeError("pip install failed: %s" % pkg_spec)
    log("pip ok: %s" % pkg_spec)

def have(mod):
    return subprocess.run("%s -c \"import %s\"" % (sys.executable, mod), shell=True, capture_output=True).returncode == 0

for _m, _pkg in [("sklearn", "scikit-learn==1.3.2"), ("pandas", "pandas<2.2.0"), ("matplotlib", "matplotlib<3.9.0"), ("tqdm", "tqdm")]:
    if not have(_m):
        pip_install(_pkg)
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
log("device=%s | torch=%s" % (DEVICE, torch.__version__))

In [ ]:
# ==================== Load dataset (windows + packed sessions) ===============
windows_df = pd.read_csv(WINDOWS_INDEX_CSV)
windows_df = windows_df[windows_df.expressed_label_id >= 0].copy()
windows_df["expressed_label_id"] = windows_df.expressed_label_id.astype(int)
log("trainable windows: %d (of %d total in Phase 1)" % (len(windows_df), int(manifest["counts"]["windows_total"])))
print(windows_df.expressed_label.value_counts().reindex(list(EMOTION_NAMES.values())).to_string())
assert len(windows_df) >= 50, "Too few trainable windows (%d). Collect more sessions or clear the review queue in Phase 1." % len(windows_df)

_session_cache = {}
def load_session(sid):
    sid = str(sid)
    if sid not in _session_cache:
        if len(_session_cache) > 6:
            _session_cache.clear()
        _session_cache[sid] = torch.load(SESSIONS_DIR / (sid + ".pt"), map_location="cpu", weights_only=False)
    return _session_cache[sid]

def window_hidden(row):
    s = load_session(row.session_id)
    return s["helium"][int(row.start_step):int(row.end_step)].float()   # [T<=100, 4096]

def soft_target(row, source=SOFT_SOURCE):
    col = source + "_scores_json"
    v = row.get(col)
    if not isinstance(v, str) or not v:
        return None
    arr = np.asarray(json.loads(v), dtype=np.float32)
    return arr if arr.shape == (NUM_EMOTIONS,) and arr.sum() > 0 else None

# quick integrity check of one window per session
for _sid in windows_df.session_id.unique()[:5]:
    _r = windows_df[windows_df.session_id == _sid].iloc[0]
    _h = window_hidden(_r)
    assert _h.ndim == 2 and _h.shape[1] == HIDDEN_DIM and _h.shape[0] >= WINDOW_STEPS // 2
log("session/window loading verified (hidden windows [T<=100, 4096]).")

## Leakage-safe splits

- **Main split:** by `session_id` (windows of one session never cross train/val/test). Sessions are short and same-scenario sessions share persona text, so session-level grouping is the honest unit.
- **Voice-held-out:** per-voice metrics on the main test set, plus optional **leave-one-voice-out** retraining (`RUN_LOVO`) to measure voice generalization -- the closest available proxy for unseen-speaker robustness (all voices are PersonaPlex's own; true user-voice independence is not the target since this head reads the *model's* hidden states, not the user's voice).

In [ ]:
# ==================== Leakage-safe splits ====================================
from sklearn.model_selection import GroupShuffleSplit

SPLIT_CSV = PHASE2_DIR / "split_assignment.csv"
if SPLIT_CSV.exists() and not FORCE_RETRAIN:
    split_map = dict(pd.read_csv(SPLIT_CSV).values)
    log("loaded existing split assignment")
else:
    _sids = windows_df.session_id.astype(str).unique().tolist()
    # stratify sessions by their dominant expressed class
    _dom = windows_df.groupby("session_id").expressed_label_id.agg(lambda s: int(s.mode().iloc[0]))
    _y = [_dom[s] for s in _sids]
    gss = GroupShuffleSplit(n_splits=1, test_size=TEST_FRACTION, random_state=SEED)
    trv, te = next(gss.split(_sids, _y, groups=_sids))
    _rest = [(_sids[i], _y[i]) for i in trv]
    gss2 = GroupShuffleSplit(n_splits=1, test_size=VAL_FRACTION / (1 - TEST_FRACTION), random_state=SEED)
    tr_i, va_i = next(gss2.split([r[0] for r in _rest], [r[1] for r in _rest], groups=[r[0] for r in _rest]))
    split_map = {}
    for i in tr_i:
        split_map[_rest[i][0]] = "train"
    for i in va_i:
        split_map[_rest[i][0]] = "val"
    for i in te:
        split_map[_sids[i]] = "test"
    pd.DataFrame({"session_id": list(split_map), "split": list(split_map.values())}).to_csv(SPLIT_CSV, index=False)
    log("new split assignment written")

windows_df["split"] = windows_df.session_id.astype(str).map(split_map)
for _s in ("train", "val", "test"):
    _d = windows_df[windows_df.split == _s]
    log("split %-5s: %4d windows | %s" % (_s, len(_d), _d.expressed_label.value_counts().to_dict()))
_voices = sorted(windows_df.voice.unique().tolist())
log("voices: %s" % _voices)

In [ ]:
# ==================== Unified head architectures =============================
import torch.nn as nn
import torch.nn.functional as F

class MLPHead(nn.Module):
    def __init__(self, d_in=HIDDEN_DIM, hidden=HEAD_HIDDEN, n_cls=NUM_EMOTIONS):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_in, hidden), nn.GELU(), nn.Dropout(0.2), nn.Linear(hidden, n_cls))
    def forward(self, x, lengths=None):
        if x.ndim == 3:
            if lengths is not None:
                m = (torch.arange(x.shape[1], device=x.device)[None, :] < lengths[:, None]).float().unsqueeze(-1)
                x = (x * m).sum(1) / m.sum(1).clamp(min=1)
            else:
                x = x.mean(1)
        return self.net(x)

class GRUHead(nn.Module):
    def __init__(self, d_in=HIDDEN_DIM, hidden=HEAD_HIDDEN, n_cls=NUM_EMOTIONS):
        super().__init__()
        self.proj = nn.Linear(d_in, hidden)
        self.gru = nn.GRU(hidden, hidden, batch_first=True)
        self.out = nn.Linear(hidden, n_cls)
    def forward(self, x, lengths=None):
        h = self.proj(x)
        h, _ = self.gru(h)
        if lengths is not None:
            idx = (lengths - 1).clamp(min=0).view(-1, 1, 1).expand(-1, 1, h.shape[-1])
            h = h.gather(1, idx)[:, 0]
        else:
            h = h[:, -1]
        return self.out(h)

class TinyTransformerHead(nn.Module):
    def __init__(self, d_in=HIDDEN_DIM, d_model=HEAD_HIDDEN, n_heads=4, n_layers=2, n_cls=NUM_EMOTIONS, max_len=128):
        super().__init__()
        self.proj = nn.Linear(d_in, d_model)
        self.pos = nn.Parameter(torch.zeros(1, max_len, d_model))
        layer = nn.TransformerEncoderLayer(d_model, n_heads, d_model * 4, dropout=0.1, batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(layer, n_layers)
        self.out = nn.Linear(d_model, n_cls)
    def forward(self, x, lengths=None):
        h = self.proj(x) + self.pos[:, :x.shape[1]]
        pad = None
        if lengths is not None:
            pad = torch.arange(x.shape[1], device=x.device)[None, :] >= lengths[:, None]
        h = self.enc(h, src_key_padding_mask=pad)
        if pad is not None:
            m = (~pad).float().unsqueeze(-1)
            h = (h * m).sum(1) / m.sum(1).clamp(min=1)
        else:
            h = h.mean(1)
        return self.out(h)

def build_head(head_type=HEAD_TYPE):
    if head_type == "mlp":
        return MLPHead()
    if head_type == "tinyth":
        return TinyTransformerHead()
    return GRUHead()

_n_params = sum(p.numel() for p in build_head().parameters())
log("head '%s': %.1fK params (%.2f MB fp32, %.2f MB fp16)" % (HEAD_TYPE, _n_params / 1e3, _n_params * 4 / 1e6, _n_params * 2 / 1e6))

class WindowDataset(torch.utils.data.Dataset):
    def __init__(self, df):
        self.rows = list(df.itertuples())
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, i):
        r = self.rows[i]
        h = window_hidden(r)
        y = int(r.expressed_label_id)
        soft = soft_target(r._asdict()) if LOSS_MODE in ("soft", "hard+soft") else None
        return h, y, (torch.from_numpy(soft) if soft is not None else None)

def collate(batch):
    Ts = [b[0].shape[0] for b in batch]
    Tmax = max(Ts)
    xs = torch.zeros(len(batch), Tmax, HIDDEN_DIM)
    lengths = torch.tensor(Ts, dtype=torch.long)
    ys = torch.tensor([b[1] for b in batch], dtype=torch.long)
    softs = None
    if batch[0][2] is not None:
        softs = torch.stack([b[2] if b[2] is not None else torch.zeros(NUM_EMOTIONS) for b in batch])
    for i, b in enumerate(batch):
        xs[i, :Ts[i]] = b[0]
    return xs, lengths, ys, softs

def make_loader(split, shuffle=False):
    df = windows_df[windows_df.split == split]
    return torch.utils.data.DataLoader(WindowDataset(df), batch_size=BATCH_SIZE, shuffle=shuffle, collate_fn=collate)
log("dataset/loader machinery ready.")

In [ ]:
# ==================== Training loop (checkpoint/resume) ======================
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

def evaluate(model, loader):
    model.eval()
    preds, gold, probs = [], [], []
    with torch.no_grad():
        for xs, lengths, ys, _ in loader:
            logits = model(xs.to(DEVICE), lengths.to(DEVICE))
            probs.append(torch.softmax(logits, -1).cpu())
            preds.append(logits.argmax(-1).cpu())
            gold.append(ys)
    preds, gold, probs = torch.cat(preds).numpy(), torch.cat(gold).numpy(), torch.cat(probs).numpy()
    return gold, preds, probs

def metrics_dict(gold, preds):
    emo = gold > 0
    return dict(
        acc=round(float(accuracy_score(gold, preds)), 4),
        macro_f1=round(float(f1_score(gold, preds, average="macro")), 4),
        per_class_f1={EMOTION_NAMES[i]: round(float(f1_score(gold == i, preds == i)), 4) for i in range(NUM_EMOTIONS)},
        binary_neutral_vs_emo_f1=round(float(f1_score(emo, preds > 0)) if emo.any() and (~emo).any() else float("nan"), 4),
        confusion=confusion_matrix(gold, preds, labels=list(range(NUM_EMOTIONS))).tolist(),
        n=int(len(gold)),
    )

def train_head(model, train_df, val_df, tag="main", epochs=EPOCHS):
    global windows_df
    _backup = windows_df
    try:
        windows_df = pd.concat([train_df.assign(split="train"), val_df.assign(split="val")])
        tr_loader = make_loader("train", shuffle=True)
        va_loader = make_loader("val")
        model = model.to(DEVICE)
        ys_all = torch.tensor([int(r.expressed_label_id) for r in train_df.itertuples()])
        counts = torch.bincount(ys_all, minlength=NUM_EMOTIONS).float()
        weights = (counts.sum() / counts.clamp(min=1)).clamp(max=10.0).to(DEVICE)
        opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
        ckpt = PHASE2_DIR / ("train_checkpoint_%s.pt" % tag)
        start_ep, best_f1, best_state, bad = 0, -1.0, None, 0
        if ckpt.exists() and not FORCE_RETRAIN:
            st = torch.load(ckpt, map_location=DEVICE, weights_only=False)
            model.load_state_dict(st["model"]); opt.load_state_dict(st["opt"])
            start_ep, best_f1, best_state, bad = st["epoch"] + 1, st["best_f1"], st["best_state"], st["bad"]
            log("[%s] resumed at epoch %d (best val F1 %.3f)" % (tag, start_ep, best_f1))
        for ep in range(start_ep, epochs):
            model.train()
            tot_loss, nb = 0.0, 0
            for xs, lengths, ys, softs in tr_loader:
                logits = model(xs.to(DEVICE), lengths.to(DEVICE))
                if LOSS_MODE == "soft" and softs is not None:
                    loss = F.kl_div(F.log_softmax(logits, -1), softs.to(DEVICE), reduction="batchmean")
                elif LOSS_MODE == "hard+soft" and softs is not None:
                    loss = 0.5 * F.cross_entropy(logits, ys.to(DEVICE), weight=weights) + 0.5 * F.kl_div(F.log_softmax(logits, -1), softs.to(DEVICE), reduction="batchmean")
                else:
                    loss = F.cross_entropy(logits, ys.to(DEVICE), weight=weights)
                opt.zero_grad(); loss.backward(); opt.step()
                tot_loss += float(loss); nb += 1
            vg, vp, _ = evaluate(model, va_loader)
            vf1 = f1_score(vg, vp, average="macro")
            log("[%s] ep %d loss=%.4f val_macroF1=%.3f" % (tag, ep, tot_loss / max(nb, 1), vf1))
            if vf1 > best_f1:
                best_f1, bad = vf1, 0
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            else:
                bad += 1
            torch.save({"epoch": ep, "model": model.state_dict(), "opt": opt.state_dict(),
                        "best_f1": best_f1, "best_state": best_state, "bad": bad}, ckpt)
            if bad >= PATIENCE:
                log("[%s] early stop at ep %d" % (tag, ep))
                break
        if best_state is not None:
            model.load_state_dict(best_state)
        return model, best_f1
    finally:
        windows_df = _backup

log("training machinery ready.")

## Training the unified head

Trains on **expressed labels** (human-corrected where available) with class-weighted CE (or teacher soft-label distillation per `LOSS_MODE`). Checkpoints every epoch to `train_checkpoint_main.pt` -- interrupt and re-run freely; training resumes. The final artifact keeps the best-validation weights.

In [ ]:
# ==================== Main training run ======================================
_train_df = windows_df[windows_df.split == "train"]
_val_df = windows_df[windows_df.split == "val"]
_test_df = windows_df[windows_df.split == "test"]

model = build_head()
if MODEL_PATH.exists() and not FORCE_RETRAIN:
    _st = torch.load(MODEL_PATH, map_location=DEVICE, weights_only=False)
    model.load_state_dict(_st["model"])
    model = model.to(DEVICE)
    best_val_f1 = _st.get("best_val_f1", float("nan"))
    log("loaded existing trained head from %s (best val F1 %.3f) -- skipping training (FORCE_RETRAIN=True to redo)" % (MODEL_PATH, best_val_f1))
else:
    model, best_val_f1 = train_head(model, _train_df, _val_df, tag="main")
    torch.save({"model": model.state_dict(), "best_val_f1": best_val_f1,
                "head_type": HEAD_TYPE, "input_type": INPUT_TYPE}, MODEL_PATH)
    log("saved trained head -> %s (best val F1 %.3f)" % (MODEL_PATH, best_val_f1))

# ---- test evaluation ----
te_loader = make_loader("test")
gold, preds, probs = evaluate(model, te_loader)
test_metrics = metrics_dict(gold, preds)
log("TEST: acc=%.3f macroF1=%.3f neutral-vs-emo F1=%.3f" % (test_metrics["acc"], test_metrics["macro_f1"], test_metrics["binary_neutral_vs_emo_f1"]))
print(json.dumps({k: v for k, v in test_metrics.items() if k != "confusion"}, indent=2))

# ---- per-voice metrics (generalization visibility) ----
per_voice = {}
_test_with_voice = _test_df.copy()
for _v in _voices:
    _m = (_test_with_voice.voice == _v).values
    if _m.sum() < 5:
        continue
    per_voice[_v] = metrics_dict(gold[_m], preds[_m])
log("per-voice test macroF1: %s" % {v: d["macro_f1"] for v, d in per_voice.items()})

# confusion plot
fig, ax = plt.subplots(figsize=(6, 5))
cm = np.array(test_metrics["confusion"])
ax.imshow(cm, cmap="Blues")
ax.set_title("unified head -- test confusion (acc=%.2f F1=%.2f)" % (test_metrics["acc"], test_metrics["macro_f1"]))
ax.set_xticks(range(NUM_EMOTIONS), [EMOTION_NAMES[i] for i in range(NUM_EMOTIONS)], rotation=45, ha="right")
ax.set_yticks(range(NUM_EMOTIONS), [EMOTION_NAMES[i] for i in range(NUM_EMOTIONS)])
for i in range(NUM_EMOTIONS):
    for j in range(NUM_EMOTIONS):
        if cm[i, j]:
            ax.text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=9)
fig.tight_layout(); fig.savefig(str(PLOTS_DIR / "test_confusion.png"), dpi=110); plt.close(fig)
log("saved plots/test_confusion.png")

In [ ]:
# ============== Leave-one-voice-out generalization (optional) ================
lovo_results = {}
if RUN_LOVO and len(_voices) > 1:
    for _held in _voices:
        _tr = windows_df[(windows_df.split.isin(["train", "val"])) & (windows_df.voice != _held)]
        _va = windows_df[(windows_df.split == "val") & (windows_df.voice != _held)]
        if len(_tr) < 20 or len(_va) < 5:
            log("LOVO skip %s (insufficient data)" % _held)
            continue
        _m = build_head()
        _m, _bf = train_head(_m, _tr, _va, tag="lovo_%s" % Path(str(_held)).stem, epochs=max(15, EPOCHS // 2))
        _te_voice = windows_df[(windows_df.split == "test") & (windows_df.voice == _held)]
        if len(_te_voice) < 5:
            continue
        _bk = windows_df
        windows_df = _te_voice.assign(split="test")
        _g, _p, _ = evaluate(_m, make_loader("test"))
        windows_df = _bk
        lovo_results[_held] = metrics_dict(_g, _p)
        log("LOVO held-out=%s: acc=%.3f macroF1=%.3f (n=%d)" % (_held, lovo_results[_held]["acc"], lovo_results[_held]["macro_f1"], lovo_results[_held]["n"]))
    save_json(lovo_results, PHASE2_DIR / "lovo_results.json")
else:
    log("LOVO skipped (RUN_LOVO=%s, voices=%d)" % (RUN_LOVO, len(_voices)))

In [ ]:
# ==================== Calibration (temperature scaling) ======================
def ece(probs, gold, n_bins=10):
    conf = probs.max(axis=1); pred = probs.argmax(axis=1)
    bins = np.linspace(0, 1, n_bins + 1)
    out = 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.sum():
            out += (m.sum() / len(gold)) * abs(float((pred[m] == gold[m]).mean()) - float(conf[m].mean()))
    return float(out)

# fit temperature on validation logits
va_loader = make_loader("val")
def collect_logits(loader):
    model.eval()
    L, G = [], []
    with torch.no_grad():
        for xs, lengths, ys, _ in loader:
            L.append(model(xs.to(DEVICE), lengths.to(DEVICE)).cpu())
            G.append(ys)
    return torch.cat(L), torch.cat(G)
va_logits, va_gold = collect_logits(va_loader)
te_logits, te_gold = collect_logits(te_loader)

temp = torch.nn.Parameter(torch.ones(1))
_opt = torch.optim.LBFGS([temp], lr=0.05, max_iter=60)
def _closure():
    _opt.zero_grad()
    loss = F.cross_entropy(va_logits / temp.clamp(min=0.05), va_gold)
    loss.backward()
    return loss
_opt.step(_closure)
TEMPERATURE = float(temp.clamp(min=0.05).item())
ece_before = ece(torch.softmax(te_logits, -1).numpy(), te_gold.numpy())
ece_after = ece(torch.softmax(te_logits / TEMPERATURE, -1).numpy(), te_gold.numpy())
log("temperature scaling: T=%.3f | test ECE %.4f -> %.4f" % (TEMPERATURE, ece_before, ece_after))

## Live-contract validation

The head must run inside the live pipeline without disturbing it. We simulate the exact live geometry -- a 100-step (8 s) hidden-state window (the live deque) at 12.5 Hz -- and measure:

- **Latency** per inference (p50/p95) on GPU (fp32 + fp16) and CPU, against a 10 ms target (the 80 ms step budget belongs to PersonaPlex + renderer; the head must be a rounding error).
- **VRAM** of the loaded head.
- **Streaming simulation**: sliding the window one step at a time over a test session, confirming the head produces a stable emotion trajectory (the live server publishes `emotion_id` on a cadence, with confidence).

In [ ]:
# ==================== Live-contract validation ===============================
model.eval()
_n_params = sum(p.numel() for p in model.parameters())

def bench(x, lengths, mdl, iters=200):
    with torch.no_grad():
        for _ in range(20):
            mdl(x, lengths)
        if x.is_cuda:
            torch.cuda.synchronize()
        ts = []
        for _ in range(iters):
            t0 = time.perf_counter()
            mdl(x, lengths)
            if x.is_cuda:
                torch.cuda.synchronize()
            ts.append((time.perf_counter() - t0) * 1e3)
    return float(np.percentile(ts, 50)), float(np.percentile(ts, 95))

x_gpu = torch.randn(1, WINDOW_STEPS, HIDDEN_DIM, device=DEVICE)
l_gpu = torch.tensor([WINDOW_STEPS], device=DEVICE)
p50_gpu, p95_gpu = bench(x_gpu, l_gpu, model)

model_half = build_head().to(DEVICE)
model_half.load_state_dict(model.state_dict())
model_half = model_half.half().eval()
p50_fp16, p95_fp16 = bench(x_gpu.half(), l_gpu, model_half)

model_cpu = build_head()
model_cpu.load_state_dict(model.state_dict())
model_cpu = model_cpu.float().eval()
x_cpu = torch.randn(1, WINDOW_STEPS, HIDDEN_DIM)
l_cpu = torch.tensor([WINDOW_STEPS])
p50_cpu, p95_cpu = bench(x_cpu, l_cpu, model_cpu, iters=50)

if DEVICE == "cuda":
    torch.cuda.reset_peak_memory_stats()
    with torch.no_grad():
        model(x_gpu, l_gpu)
    head_vram_mb = torch.cuda.max_memory_allocated() / 1e6
else:
    head_vram_mb = 0.0

latency = {
    "gpu_fp32": {"p50_ms": round(p50_gpu, 3), "p95_ms": round(p95_gpu, 3)},
    "gpu_fp16": {"p50_ms": round(p50_fp16, 3), "p95_ms": round(p95_fp16, 3)},
    "cpu_fp32": {"p50_ms": round(p50_cpu, 3), "p95_ms": round(p95_cpu, 3)},
    "params": int(_n_params), "head_vram_mb": round(float(head_vram_mb), 1),
    "budget_ms": LIVE_BUDGET_MS,
    "pass": bool(p95_gpu < LIVE_BUDGET_MS),
}
log("latency: %s" % json.dumps(latency))
assert latency["pass"], "Head is too slow for the live budget -- pick a smaller head (mlp/gru) or fewer layers."

# streaming simulation: slide the 100-step window over a real test session
_sid = _test_df.session_id.iloc[0]
_s = load_session(_sid)
_T = int(_s["helium"].shape[0])
_traj = []
with torch.no_grad():
    for _t in range(WINDOW_STEPS, _T + 1, 5):
        _w = _s["helium"][_t - WINDOW_STEPS:_t].float().unsqueeze(0).to(DEVICE)
        _l = torch.tensor([WINDOW_STEPS], device=DEVICE)
        _p = torch.softmax(model(_w, _l) / TEMPERATURE, -1)[0].cpu().numpy()
        _traj.append((_t, int(_p.argmax()), float(_p.max())))
log("streaming simulation over session %s (%d steps): %d window evaluations" % (_sid, _T, len(_traj)))
log("emotion trajectory (step, label, conf) sample: %s" % [(t, EMOTION_NAMES[e], round(c, 2)) for t, e, c in _traj[:8]])

In [ ]:
# ==================== Export + live integration contract =====================
# The exported config is the SINGLE source of truth the live server reads.
config = {
    "artifact": "unified_emotion_head.pt",
    "head_type": HEAD_TYPE,
    "input_type": INPUT_TYPE,
    "hidden_dim": HIDDEN_DIM,
    "head_hidden": HEAD_HIDDEN,
    "num_emotions": NUM_EMOTIONS,
    "emotion_names": EMOTION_NAMES,
    "window_steps": WINDOW_STEPS,
    "lookahead_steps": LOOKAHEAD_STEPS,
    "step_hz": STEP_HZ,
    "temperature": TEMPERATURE,
    "trained_on": {"dataset_manifest": str(MANIFEST_JSON), "loss_mode": LOSS_MODE,
                   "n_train": int(len(_train_df)), "n_val": int(len(_val_df)), "n_test": int(len(_test_df))},
    "test_metrics": test_metrics,
    "per_voice_test": per_voice,
    "ece": {"before": round(ece_before, 4), "after": round(ece_after, 4)},
    "latency": latency,
    "live_integration": {
        "replace": "auto_emotion.SpeechEmotionClassifier (user-microphone emotion2vec) as the source of emotion_id",
        "keep": "EmotionConditioner -> FMT CFG -> renderer path unchanged; only the emotion_id producer changes",
        "feed": "the same helium hidden deque the AH server already maintains (100 steps @ 12.5 Hz) + 6-step lookahead",
        "publish": "emotion_id = argmax softmax(head(window)/temperature); publish confidence alongside; smooth with the existing EMA/hysteresis logic",
    },
}
save_json(config, CONFIG_PATH)
# re-save the model with full metadata
torch.save({"model": model.state_dict(), "best_val_f1": float(best_val_f1),
            "head_type": HEAD_TYPE, "input_type": INPUT_TYPE, "temperature": TEMPERATURE,
            "config_path": str(CONFIG_PATH)}, MODEL_PATH)
log("exported %s and %s" % (MODEL_PATH, CONFIG_PATH))

LIVE_SNIPPET = PHASE2_DIR / "live_integration_example.py"
LIVE_SNIPPET.write_text(
    "# Drop-in live integration example (reference only -- adapt to the AH server).\n"
    "# Replaces the user-voice emotion2vec emotion source with the unified head.\n"
    "import json, torch\n"
    "from pathlib import Path\n\n"
    "class UnifiedPersonaEmotion:\n"
    "    def __init__(self, model_path, config_path, device='cuda'):\n"
    "        cfg = json.loads(Path(config_path).read_text())\n"
    "        self.cfg = cfg\n"
    "        st = torch.load(model_path, map_location=device, weights_only=False)\n"
    "        import torch.nn as nn\n"
    "        d, h, n = cfg['hidden_dim'], cfg['head_hidden'], cfg['num_emotions']\n"
    "        class GRUHead(nn.Module):\n"
    "            def __init__(self):\n"
    "                super().__init__()\n"
    "                self.proj = nn.Linear(d, h); self.gru = nn.GRU(h, h, batch_first=True); self.out = nn.Linear(h, n)\n"
    "            def forward(self, x, lengths):\n"
    "                y, _ = self.gru(self.proj(x))\n"
    "                idx = (lengths - 1).clamp(min=0).view(-1, 1, 1).expand(-1, 1, y.shape[-1])\n"
    "                return self.out(y.gather(1, idx)[:, 0])\n"
    "        self.model = GRUHead().to(device).eval()\n"
    "        self.model.load_state_dict(st['model'])\n"
    "        self.T = float(cfg['temperature'])\n"
    "        self.device = device\n"
    "    @torch.no_grad()\n"
    "    def emotion_id(self, helium_window):\n"
    "        # helium_window: float tensor [100, 4096] from the live deque (fp16 ok)\n"
    "        x = helium_window.float().unsqueeze(0).to(self.device)\n"
    "        lengths = torch.tensor([x.shape[1]], device=self.device)\n"
    "        p = torch.softmax(self.model(x, lengths) / self.T, -1)[0]\n"
    "        return int(p.argmax()), float(p.max())\n",
    encoding="utf-8")
log("wrote %s (reference integration snippet)" % LIVE_SNIPPET)

In [ ]:
# ==================== Final report ===========================================
results = {
    "phase": 2,
    "created": time.strftime("%Y-%m-%d %H:%M:%S"),
    "head": {"type": HEAD_TYPE, "input": INPUT_TYPE, "params": int(_n_params), "loss_mode": LOSS_MODE},
    "phase0_recommendation": PHASE0_RECO.get("input_and_head"),
    "best_val_macro_f1": float(best_val_f1),
    "test_metrics": test_metrics,
    "per_voice_test": per_voice,
    "lovo": lovo_results,
    "calibration": {"temperature": TEMPERATURE, "ece_before": round(ece_before, 4), "ece_after": round(ece_after, 4)},
    "latency": latency,
    "artifacts": {"model": str(MODEL_PATH), "config": str(CONFIG_PATH), "snippet": str(LIVE_SNIPPET)},
}
save_json(results, RESULTS_JSON)

lines = []
lines.append("# Phase 2 Report -- Unified PersonaPlex Emotion Network")
lines.append("")
lines.append("- Head: `%s` on `%s` (%d params) | loss=%s" % (HEAD_TYPE, INPUT_TYPE, _n_params, LOSS_MODE))
lines.append("- Data: %d train / %d val / %d test windows (session-level splits, no leakage)" % (len(_train_df), len(_val_df), len(_test_df)))
lines.append("- **Test: acc=%.3f, macro F1=%.3f, neutral-vs-emotional F1=%.3f**" % (test_metrics["acc"], test_metrics["macro_f1"], test_metrics["binary_neutral_vs_emo_f1"]))
lines.append("- Per-voice test macro F1: %s" % {v: d["macro_f1"] for v, d in per_voice.items()})
if lovo_results:
    lines.append("- Leave-one-voice-out macro F1: %s" % {v: d["macro_f1"] for v, d in lovo_results.items()})
lines.append("- Calibration: T=%.3f, ECE %.4f -> %.4f" % (TEMPERATURE, ece_before, ece_after))
lines.append("- Live latency: GPU fp32 p95=%.2f ms, fp16 p95=%.2f ms, CPU p95=%.2f ms (budget %.0f ms) -> PASS" % (p95_gpu, p95_fp16, p95_cpu, LIVE_BUDGET_MS))
lines.append("")
lines.append("## Per-class F1 (test)")
for name, f1 in test_metrics["per_class_f1"].items():
    lines.append("- %s: %.3f" % (name, f1))
lines.append("")
lines.append("## Live integration")
lines.append("Replace the user-microphone emotion2vec emotion source with this head; feed it the live Helium hidden deque (100 steps @ 12.5 Hz); publish emotion_id + confidence through the existing EmotionConditioner -> FM CFG path, unchanged. See unified_emotion_head_config.json and live_integration_example.py.")
REPORT_MD.write_text("\n".join(lines), encoding="utf-8")
log("wrote %s and %s" % (RESULTS_JSON, REPORT_MD))
print("\n".join(lines))

## Phase 2 complete -- the unified emotion network is ready

**Artifacts** (`/workspace/persona_emotion/phase2/`):

- `unified_emotion_head.pt` -- trained head weights (+ metadata)
- `unified_emotion_head_config.json` -- deployment contract (architecture, window geometry, temperature, metrics)
- `live_integration_example.py` -- reference drop-in for the live server
- `phase2_results.json` / `phase2_report.md` -- full evaluation record

**Live swap (the only change to the live system):** the AH server already maintains the Helium hidden deque (100 steps at 12.5 Hz + lookahead). Where the current pipeline runs `SpeechEmotionClassifier` on the **user microphone** to obtain `emotion_id`, run this head on the **reply hidden-state deque** instead and publish its `emotion_id` + confidence (temperature-calibrated, with the existing EMA/hysteresis smoothing). The downstream `EmotionConditioner` -> FMT CFG -> renderer path is untouched.

**If metrics are weak:** revisit Phase 1 (more sessions, clearer review queue), try `LOSS_MODE="soft"` or `"hard+soft"` distillation, or switch `HEAD_TYPE`/`INPUT_TYPE` to whatever Phase 0 probed best. Everything is resumable; only the final training re-runs.